# Hệ Thống Phát Hiện Bó Mỳ Chũ & Dị Vật Bề Mặt Bằng YOLOv8
**Chủ đề:** Ứng dụng Thị giác máy tính (Computer Vision) tự động định vị sản phẩm và kiểm định chất lượng bề mặt Mỳ Chũ  
**Bài toán:** Object Detection 2 lớp (`0: noodle_bundle`, `1: foreign_object`) với kiểm soát chống rò rỉ dữ liệu theo cụm bó mì vật lý (Bundle-based Group Split).

---

## CELL 1: Cài đặt thư viện & Kiểm tra môi trường phần cứng (GPU)

In [ ]:
!pip install -q ultralytics

import os
import shutil
import zipfile
import yaml
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torch
import ultralytics
from ultralytics import YOLO

ultralytics.checks()
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device      : {torch.cuda.get_device_name(0)}")

## CELL 2: Thu thập & Kiểm định toàn vẹn 1.000 ảnh + 1.000 nhãn từ `/kaggle/input`
Cell này tự động quét toàn bộ thư mục `/kaggle/input` để ghép cặp 1.000 bức ảnh gốc (`My Chu Raw Image Dataset V1`) với 1.000 file nhãn `.txt` từ gói `my_chu_labels_1000_2classes.zip`.

In [ ]:
WORK_DIR = Path("/kaggle/working")
YOLO_ROOT = WORK_DIR / "my_chu_yolo_dataset"

CLASS_NAMES = {
    0: "noodle_bundle",
    1: "foreign_object"
}

img_map = {}  # {stem: Path}
lbl_map = {}  # {stem: str_content}

for root, dirs, files in os.walk("/kaggle/input"):
    root_p = Path(root)
    for fname in files:
        fp = root_p / fname
        stem = fp.stem
        ext = fp.suffix.lower()

        if ext in {".jpg", ".jpeg", ".png"} and (stem.startswith("MC_C_") or stem.startswith("MC_FO_")):
            img_map[stem] = fp
        elif ext == ".txt" and (stem.startswith("MC_C_") or stem.startswith("MC_FO_")):
            lbl_map[stem] = fp.read_text(encoding="utf-8")
        elif ext == ".zip":
            with zipfile.ZipFile(fp, "r") as zf:
                for zname in zf.namelist():
                    zstem = Path(zname).stem
                    if zname.endswith(".txt") and (zstem.startswith("MC_C_") or zstem.startswith("MC_FO_")):
                        lbl_map[zstem] = zf.read(zname).decode("utf-8")

clean_imgs = [k for k in img_map if k.startswith("MC_C_")]
fo_imgs = [k for k in img_map if k.startswith("MC_FO_")]

print("=" * 65)
print("BÁO CÁO QUÉT DỮ LIỆU ĐẦU VÀO (/kaggle/input)")
print("=" * 65)
print(f" - Tổng số ảnh tìm thấy      : {len(img_map)} / 1000 (Clean: {len(clean_imgs)}, FO: {len(fo_imgs)})")
print(f" - Tổng số file nhãn (.txt)  : {len(lbl_map)} / 1000")
print(f" - Khớp cặp 1-1 Ảnh & Nhãn   : {set(img_map.keys()) == set(lbl_map.keys())}")
print("=" * 65)

assert len(img_map) == 1000, f"Thiếu ảnh! Chỉ tìm thấy {len(img_map)} ảnh."
assert len(lbl_map) == 1000, f"Thiếu nhãn! Chỉ tìm thấy {len(lbl_map)} file .txt."
assert set(img_map.keys()) == set(lbl_map.keys()), "Tên file ảnh và nhãn không khớp 1-1!"

## CELL 3: Kiểm định chất lượng nhãn (Annotation QA Gate) & Phân tích dữ liệu (EDA)
Kiểm tra nghiêm ngặt toàn bộ 1.500 bounding boxes:
- Mỗi ảnh `MC_C_xxxx` phải có đúng **1 box** thuộc lớp `0: noodle_bundle`.
- Mỗi ảnh `MC_FO_xxxx` phải có đúng **2 boxes**: `0: noodle_bundle` và `1: foreign_object`.
- Toàn bộ tọa độ chuẩn hóa $(x_c, y_c, w, h)$ phải nằm trong miền hợp lệ $(0, 1]$.

In [ ]:
records = []
qa_errors = []

for stem in sorted(lbl_map.keys()):
    group = "clean" if stem.startswith("MC_C_") else "foreign_object"
    lines = [ln.strip() for ln in lbl_map[stem].splitlines() if ln.strip()]

    if group == "clean" and len(lines) != 1:
        qa_errors.append((stem, f"Clean image must have 1 box, got {len(lines)}"))
    elif group == "foreign_object" and len(lines) != 2:
        qa_errors.append((stem, f"FO image must have 2 boxes, got {len(lines)}"))

    for ln in lines:
        parts = ln.split()
        if len(parts) != 5:
            qa_errors.append((stem, f"Malformed line: {ln}"))
            continue
        cid = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:])
        if cid not in (0, 1) or not (0.0 < xc <= 1.0 and 0.0 < yc <= 1.0 and 0.0 < bw <= 1.0 and 0.0 < bh <= 1.0):
            qa_errors.append((stem, f"Invalid coordinates: {ln}"))

        records.append({
            "stem": stem,
            "group": group,
            "class_id": cid,
            "class_name": CLASS_NAMES[cid],
            "xc": xc,
            "yc": yc,
            "bw": bw,
            "bh": bh,
            "area_ratio": bw * bh
        })

df_boxes = pd.DataFrame(records)
print(f"[QA GATE] Tổng số Bounding Boxes đã kiểm tra: {len(df_boxes)} | Số lỗi phát hiện: {len(qa_errors)}")
assert len(qa_errors) == 0, f"Phát hiện lỗi nhãn: {qa_errors[:5]}"

summary = df_boxes.groupby(["class_id", "class_name"])["area_ratio"].agg(["count", "mean", "min", "max"])
display(summary)

# Vẽ biểu đồ phân bố diện tích và vị trí tâm của Bó mì vs Dị vật
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for cid, cname, color in [(0, "noodle_bundle", "#00C853"), (1, "foreign_object", "#FF1744")]:
    sub = df_boxes[df_boxes["class_id"] == cid]
    axes[0].hist(sub["area_ratio"] * 100, bins=30, alpha=0.65, label=f"{cid}: {cname} (n={len(sub)})", color=color)
    axes[1].scatter(sub["xc"], sub["yc"], alpha=0.35, s=18, label=f"{cid}: {cname}", color=color)

axes[0].set_title("Phân bố Tỷ lệ diện tích Box (% diện tích ảnh)", fontweight="bold")
axes[0].set_xlabel("% Diện tích khung hình")
axes[0].set_ylabel("Số lượng Box")
axes[0].legend()

axes[1].set_title("Bản đồ phân bố Tâm đối tượng (x_center, y_center)", fontweight="bold")
axes[1].set_xlim(0, 1)
axes[1].set_ylim(1, 0)
axes[1].set_xlabel("x_center (chuẩn hóa)")
axes[1].set_ylabel("y_center (chuẩn hóa)")
axes[1].legend()
plt.tight_layout()
plt.show()

## CELL 4: Trực quan hóa mẫu dữ liệu (Visual Inspection Grid)
Hiển thị ngẫu nhiên 4 ảnh `clean` và 4 ảnh `foreign_object` cùng với nhãn Ground Truth để xác nhận trực quan trước khi huấn luyện.

In [ ]:
sample_clean = ["MC_C_0001", "MC_C_0060", "MC_C_0250", "MC_C_0500"]
sample_fo = ["MC_FO_0001", "MC_FO_0394", "MC_FO_0421", "MC_FO_0500"]

fig, axes = plt.subplots(2, 4, figsize=(18, 10))

for row_idx, sample_list in enumerate([sample_clean, sample_fo]):
    for col_idx, stem in enumerate(sample_list):
        ax = axes[row_idx, col_idx]
        img = Image.open(img_map[stem]).convert("RGB")
        W, H = img.size
        ax.imshow(img)

        for ln in lbl_map[stem].splitlines():
            if not ln.strip():
                continue
            cid, xc, yc, bw, bh = map(float, ln.split())
            x1 = (xc - bw / 2.0) * W
            y1 = (yc - bh / 2.0) * H
            w_px = bw * W
            h_px = bh * H

            color = "#00E676" if int(cid) == 0 else "#FF1744"
            label_txt = "0: noodle_bundle" if int(cid) == 0 else "1: foreign_object"
            rect = patches.Rectangle((x1, y1), w_px, h_px, linewidth=2.5, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(
                x1, max(35, y1 - 10), label_txt,
                color="black" if int(cid) == 0 else "white",
                fontsize=8, fontweight="bold",
                bbox=dict(facecolor=color, alpha=0.85, pad=2)
            )
        ax.set_title(f"{stem}.jpg", fontweight="bold")
        ax.axis("off")

plt.suptitle("KIỂM CHỨNG TRỰC QUAN GROUND TRUTH (Xanh lá = Bó mì [0], Đỏ = Dị vật [1])", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## CELL 5: Chia tập Train / Val / Test theo Cụm Bó Mì (Chống Data Leakage) & Xuất `data.yaml`
- **Train (70% - 700 ảnh):** Ảnh `0001 -> 0350` của mỗi nhóm (Cụm bó mì 1 đến 7).
- **Val (15% - 150 ảnh):** Ảnh `0351 -> 0425` của mỗi nhóm (Cụm bó mì 8 và nửa đầu 9).
- **Test (15% - 150 ảnh):** Ảnh `0426 -> 0500` của mỗi nhóm (Nửa sau cụm 9 và cụm 10).

In [ ]:
def get_bundle_and_split(stem):
    num = int(stem.split("_")[-1])
    bundle_id = min(10, (num - 1) // 50 + 1)
    if num <= 350:
        split = "train"
    elif num <= 425:
        split = "val"
    else:
        split = "test"
    return bundle_id, split

if YOLO_ROOT.exists():
    shutil.rmtree(YOLO_ROOT)

for sp in ["train", "val", "test"]:
    (YOLO_ROOT / "images" / sp).mkdir(parents=True, exist_ok=True)
    (YOLO_ROOT / "labels" / sp).mkdir(parents=True, exist_ok=True)

split_summary = {
    "train": {"clean": 0, "foreign_object": 0, "bundles": set()},
    "val":   {"clean": 0, "foreign_object": 0, "bundles": set()},
    "test":  {"clean": 0, "foreign_object": 0, "bundles": set()},
}

for stem in sorted(img_map.keys()):
    bundle_id, sp = get_bundle_and_split(stem)
    grp = "clean" if stem.startswith("MC_C_") else "foreign_object"

    src_img = img_map[stem]
    dst_img = YOLO_ROOT / "images" / sp / src_img.name
    dst_lbl = YOLO_ROOT / "labels" / sp / f"{stem}.txt"

    shutil.copy2(src_img, dst_img)
    dst_lbl.write_text(lbl_map[stem].strip() + "\n", encoding="utf-8")

    split_summary[sp][grp] += 1
    split_summary[sp]["bundles"].add(f"B{bundle_id:02d}")

data_yaml = {
    "path": str(YOLO_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": CLASS_NAMES
}

yaml_path = YOLO_ROOT / "data.yaml"
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("=" * 72)
print("HOÀN TẤT XUẤT DATASET CHUẨN ULTRALYTICS YOLO")
print("=" * 72)
for sp in ["train", "val", "test"]:
    s = split_summary[sp]
    tot = s["clean"] + s["foreign_object"]
    b_str = ", ".join(sorted(s["bundles"]))
    print(f" - {sp.upper():5s}: {tot:3d} ảnh (Clean: {s['clean']:3d} | FO: {s['foreign_object']:3d}) | Cụm bó mì: {b_str}")
print(f" - File cấu hình: {yaml_path}")
print("=" * 72)

## CELL 6: Huấn luyện mô hình YOLOv8 (`yolov8s.pt`)
Sử dụng độ phân giải `imgsz=640` (hoặc `800` để tăng độ nhạy với dị vật nhỏ), tối ưu hóa với Early Stopping (`patience=15`).

In [ ]:
model = YOLO("yolov8s.pt")

train_results = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    optimizer="auto",
    seed=42,
    project="/kaggle/working/runs",
    name="my_chu_yolov8s",
    exist_ok=True
)

## CELL 7: Đánh giá độc lập trên tập Test (150 ảnh Hold-out) & Xuất bảng chỉ số
Đánh giá hiệu năng thực tế trên tập `test` (những bó mì mô hình chưa từng nhìn thấy trong quá trình Train/Val) và in ra Precision, Recall, mAP@50, mAP@50-95 cho từng lớp.

In [ ]:
best_weight_path = Path("/kaggle/working/runs/my_chu_yolov8s/weights/best.pt")
best_model = YOLO(str(best_weight_path))

test_metrics = best_model.val(
    data=str(yaml_path),
    split="test",
    imgsz=640,
    batch=16,
    project="/kaggle/working/runs",
    name="my_chu_test_eval",
    exist_ok=True
)

print("\n" + "=" * 65)
print("KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP (150 ẢNH)")
print("=" * 65)
print(f" - mAP@50 (All classes)    : {test_metrics.box.map50:.4f}")
print(f" - mAP@50-95 (All classes) : {test_metrics.box.map:.4f}")
for i, cname in CLASS_NAMES.items():
    p, r, ap50, ap = test_metrics.box.class_result(i)
    print(f" - Class {i} ({cname:14s}): Precision={p:.4f} | Recall={r:.4f} | mAP@50={ap50:.4f} | mAP@50-95={ap:.4f}")
print("=" * 65)

## CELL 8: Chạy thử nghiệm suy luận (Inference) trên tập Test & Đóng gói `best.pt` tải về Web
Hiển thị kết quả dự đoán thực tế của `best.pt` và sao chép trọng số ra thẳng `/kaggle/working/my_chu_best.pt` để dễ dàng tải về tích hợp vào ứng dụng thực tế.

In [ ]:
export_weight = WORK_DIR / "my_chu_best.pt"
shutil.copy2(best_weight_path, export_weight)
print(f"[SẴN SÀNG TẢI VỀ] Trọng số tốt nhất đã được copy ra: {export_weight} ({export_weight.stat().st_size / 1e6:.2f} MB)")

# Chạy dự đoán thử trên 4 ảnh Test có dị vật và 4 ảnh Test sạch
test_samples = [
    YOLO_ROOT / "images" / "test" / "MC_C_0450.jpg",
    YOLO_ROOT / "images" / "test" / "MC_C_0490.jpg",
    YOLO_ROOT / "images" / "test" / "MC_FO_0450.jpg",
    YOLO_ROOT / "images" / "test" / "MC_FO_0490.jpg",
]

preds = best_model.predict([str(p) for p in test_samples], conf=0.25, imgsz=640)

fig, axes = plt.subplots(1, 4, figsize=(18, 6))
for ax, res, p in zip(axes, preds, test_samples):
    plotted_bgr = res.plot()
    plotted_rgb = plotted_bgr[:, :, ::-1]
    ax.imshow(plotted_rgb)
    ax.set_title(p.name, fontweight="bold")
    ax.axis("off")
plt.suptitle("DỰ ĐOÁN THỰC TẾ CỦA MÔ HÌNH YOLOv8 TRÊN TẬP TEST", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()